In [ ]:
# INFERENCE & KAGGLE SUBMISSION
!pip install dagshub mlflow lightgbm scikit-learn cloudpickle -q

import pandas as pd
import dagshub
import mlflow
import warnings
warnings.filterwarnings('ignore')

# 1. Authenticate with DagsHub
REPO_OWNER = "ejoba22"  
REPO_NAME = "walmart-sales-forecasting"
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

# 2. Load the Production Pipeline
model_name = "Walmart_LightGBM_Production_Pipeline"
model_version = 5  
model_uri = f"models:/{model_name}/{model_version}"

print(f"Loading {model_name} (Version {model_version}) from Model Registry...")
production_pipeline = mlflow.sklearn.load_model(model_uri)

# 3. Load the raw Kaggle Test Data
base = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'
raw_test = pd.read_csv(base + 'test.csv.zip', parse_dates=['Date'])
raw_features = pd.read_csv(base + 'features.csv.zip', parse_dates=['Date'])
raw_stores = pd.read_csv(base + 'stores.csv')

raw_test_data = raw_test.merge(raw_features, on=['Store', 'Date'], how='left', suffixes=('', '_feat')).merge(raw_stores, on='Store', how='left')
raw_X_test = raw_test_data.drop(columns=['IsHoliday_feat'])

# 4. Generate Predictions
# The transformer will securely handle the history and output the exact right order.
predictions = production_pipeline.predict(raw_X_test)

# 5. Format and Save Submission File
submission = pd.DataFrame({
    'Id': raw_test['Store'].astype(str) + '_' + raw_test['Dept'].astype(str) + '_' + raw_test['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': predictions
})

submission.to_csv('submission.csv', index=False)

Accessing as tvada22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

Loading Walmart_LightGBM_Production_Pipeline (Version 5) from Model Registry...


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
